In [1]:
import json 
import sys 
from pathlib import Path
fine_tune_predictions_file = "benchmark/predictions_finetune_parakeet.json"

In [2]:
with open(fine_tune_predictions_file,"r") as file:
    contents = json.load(file)

In [ ]:
import numpy as np
from finetune.score import english_spelling_normalizer, score_wer

for content in contents:
    reference = content.get("reference")
    prediction = content.get("prediction")
    score = score_wer([reference], [prediction])
    content.update({"score":score})

In [16]:
#filter by WER greater than 0.4
threshold = 0.4
filtered_contents = []
for content in contents:
    score = content.get("score")
    if score > threshold:
        filtered_contents.append(content)
print("Total Samples Above WER({}) : {} | Percentage : {}".format(threshold, len(filtered_contents), len(filtered_contents)/len(contents)))

Total Samples Above WER(0.4) : 2819 | Percentage : 0.1627410229765616


In [33]:
import random 
from IPython.display import Audio
import soundfile as sf

sample = random.choice(filtered_contents)
reference_text = sample.get("reference")
predicted_text = sample.get("prediction")
audio_path = sample.get("audio_filepath")
score = sample.get("score")

print("Reference Text : {}".format(reference_text))
print("Predicted Text : {}".format(predicted_text))
print("Score : {}".format(score))
audio, sr = sf.read(audio_path)
Audio(audio_path)

Reference Text : bye
Predicted Text : baby
Score : 1.0


In [34]:
import noisereduce as nr

filtered_audio = nr.reduce_noise(y=audio, sr=sr)
Audio(filtered_audio, rate=sr)

In [38]:
from preprocessing.cleaning import CleanAudioComponent
import torch
preprocess = CleanAudioComponent()
new = preprocess(torch.tensor(audio))
Audio(new.cpu().numpy(), rate=sr)